## Install libraries

In [1]:
!pip install fast-langdetect
!pip install emoji
!pip install --upgrade certifi


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Imports

In [2]:
import pandas as pd
import emoji
import re
# from itertools import chain
from fast_langdetect import detect
import ssl

ssl._create_default_https_context = ssl._create_unverified_context

## Load data

In [3]:
DATA_PATH = "./data"

In [4]:
# DataFrame's with comments
df_us_comments = pd.read_csv(f"{DATA_PATH}/UScomments.csv", on_bad_lines="skip", engine="python")
df_gb_comments = pd.read_csv(f"{DATA_PATH}/GBcomments.csv", on_bad_lines="skip", engine="python")

# Comments that i collected
df_cstm_comments = pd.read_csv(f"{DATA_PATH}/CSTMcomments_all_2.csv", on_bad_lines="skip", engine="python")

# DataFrame's with videos data
df_us_videos = pd.read_csv(f"{DATA_PATH}/USvideos.csv", on_bad_lines="skip", engine="python")
df_gb_videos = pd.read_csv(f"{DATA_PATH}/GBvideos.csv", on_bad_lines="skip", engine="python")

# Videos that i collected
df_cstm_videos = pd.read_csv(f"{DATA_PATH}/CSTMvideos_all_2.csv", on_bad_lines="skip", engine="python")

us, gb: datasets from kaggle. cstm - dataset made by my own.

In [5]:
# df_cstm_comments.shape
# df_gb_comments.shape
# df_us_comments.shape

## Filter for english comments

**Functions to filter english videos and comments**

In [6]:
def filter_comments(df):
    texts = df["comment_text"].astype(str).tolist()

    langs = [detect(text)[0]["lang"] for text in texts]

    df = df.copy()
    df["lang"] = langs

    df_filtered = df[df["lang"] == "en"]

    return df_filtered

def filter_videos(df):
    titles = df["title"].astype(str).tolist()

    langs = [detect(title)[0]["lang"] for title in titles]

    df = df.copy()
    df["lang"] = langs

    df_filtered = df[df["lang"] == "en"]

    return df_filtered

Rename some columns

In [7]:
df_cstm_comments = df_cstm_comments.rename({"comment": "comment_text"}, axis=1)
df_cstm_videos = df_cstm_videos.rename({"channel": "channel_title"}, axis=1)

And futher work only with needed columns

In [8]:
comments_cols = ["video_id", "comment_text"]
videos_cols = ["video_id", "channel_title", "title"]

df_gb_comments = df_gb_comments[comments_cols]
df_us_comments = df_us_comments[comments_cols]
df_cstm_comments = df_cstm_comments[comments_cols]

df_gb_videos = df_gb_videos[videos_cols]
df_us_videos = df_us_videos[videos_cols]
df_cstm_videos = df_cstm_videos[videos_cols]

Filter for only english comments and videos

In [9]:
df_filtered_us_comments = filter_comments(df_us_comments)
df_filtered_gb_comments = filter_comments(df_gb_comments)
df_filtered_cstm_comments = filter_comments(df_cstm_comments)

df_filtered_us_videos = filter_videos(df_us_videos)
df_filtered_gb_videos = filter_videos(df_gb_videos)
df_filtered_cstm_videos = filter_videos(df_cstm_videos)

Get rid of duplicates by video_id in videos_dfs for kaggle datasets

In [10]:
df_filtered_us_videos = df_filtered_us_videos.drop_duplicates(subset=["video_id"])
df_filtered_gb_videos = df_filtered_gb_videos.drop_duplicates(subset=["video_id"])

## Merge datasets (kaggle and custom separately)

In [11]:
df_filtered_us_videos_with_comments = df_filtered_us_videos.merge(df_filtered_us_comments, on=["video_id"], how="inner")

df_filtered_gb_videos_with_comments = df_filtered_gb_videos.merge(df_filtered_gb_comments, on=["video_id"], how="inner")

df_cstm_data = df_filtered_cstm_videos.merge(df_filtered_cstm_comments, on=["video_id"], how="inner")

df_us_gb_data = pd.concat([df_filtered_us_videos_with_comments, df_filtered_gb_videos_with_comments], axis=0)

Let's look on the dataset with kaggle data

In [12]:
df_us_gb_data.head(5)

,video_id,channel_title,title,lang_x,comment_text,lang_y
0,XpVt6Z1Gjjo,Logan Paul Vlogs,1 YEAR OF VLOGGING -- HOW LOGAN PAUL CHANGED Y...,en,Logan Paul it's yo big day ‼️‼️‼️,en
1,XpVt6Z1Gjjo,Logan Paul Vlogs,1 YEAR OF VLOGGING -- HOW LOGAN PAUL CHANGED Y...,en,I've been following you from the start of your...,en
2,XpVt6Z1Gjjo,Logan Paul Vlogs,1 YEAR OF VLOGGING -- HOW LOGAN PAUL CHANGED Y...,en,Say hi to Kong and maverick for me,en
3,XpVt6Z1Gjjo,Logan Paul Vlogs,1 YEAR OF VLOGGING -- HOW LOGAN PAUL CHANGED Y...,en,MY FAN . attendance,en
4,XpVt6Z1Gjjo,Logan Paul Vlogs,1 YEAR OF VLOGGING -- HOW LOGAN PAUL CHANGED Y...,en,trending 😉,en


Let's look on the dataset with data collected by me

In [13]:
df_cstm_data.head(5)

,video_id,channel_title,title,lang_x,comment_text,lang_y
0,Rv-7LE_UY8A,We Love Animals,The cutest animal rescue you'll ever see 😍,en,Such a little HERO she deserves an award for s...,en
1,Rv-7LE_UY8A,We Love Animals,The cutest animal rescue you'll ever see 😍,en,A HUGE Hero in a pretty little princess 😊 👏 👏 👏,en
2,Rv-7LE_UY8A,We Love Animals,The cutest animal rescue you'll ever see 😍,en,❤❤❤🎉🎉🎉😊😊😊,en
3,Rv-7LE_UY8A,We Love Animals,The cutest animal rescue you'll ever see 😍,en,Yikes. Someone should of helped her. Too much ...,en
4,Rv-7LE_UY8A,We Love Animals,The cutest animal rescue you'll ever see 😍,en,"look at the innocent face, so cute",en


## Filter by len of comments

In [14]:
# Initially i fine tuned with long comments
# df_us_gb_data_filtered = df_us_gb_data[df_us_gb_data["comment_text"].str.len() >= 20].copy()
# df_cstm_data_filtered = df_cstm_data[df_cstm_data["comment_text"].str.len() >= 20].copy()

df_us_gb_data_filtered = df_us_gb_data[(df_us_gb_data["comment_text"].str.len() >= 8) & (df_us_gb_data["comment_text"].str.len() <= 300)].copy()
df_cstm_data_filtered = df_cstm_data[(df_cstm_data["comment_text"].str.len() >= 8) & (df_cstm_data["comment_text"].str.len() <= 300)].copy()

In [15]:
# df_cstm_data_filtered

## Get rid of sequence of the same emodji of length more than max_repeat

In [16]:
def trim_repeated_emoji(text, max_repeat=3):
    emojis = emoji.emoji_list(text)

    if not emojis:
        return text

    current_emoji = None
    current_emoji_cnt = None

    to_remove = set()

    for emoji_dct in emojis:
        if current_emoji is None:
            current_emoji = emoji_dct["emoji"]
            current_emoji_cnt = 1
            continue

        if emoji_dct["emoji"] != current_emoji:
            current_emoji = emoji_dct["emoji"]
            current_emoji_cnt = 1
        else:
            if 1 + current_emoji_cnt <= max_repeat:
                current_emoji_cnt += 1
            else:
                for idx in range(emoji_dct["match_start"], emoji_dct["match_end"]):
                    to_remove.add(idx)

    return ''.join(ch for i, ch in enumerate(text) if i not in to_remove)

# Save only less than or equal of max_repeat emodji in row
df_us_gb_data_filtered["comment_text"] = df_us_gb_data_filtered["comment_text"].apply(lambda x: trim_repeated_emoji(x))
df_cstm_data_filtered["comment_text"] = df_cstm_data_filtered["comment_text"].apply(lambda x: trim_repeated_emoji(x))

## Get rid of urls

In [17]:
def clean_from_urls(text):
    clean_text = re.sub(r'https?://\S+|www\.\S+', '', text).strip()

    return clean_text if len(clean_text) > 10 else None

df_us_gb_data_filtered["comment_text"] = df_us_gb_data_filtered["comment_text"].apply(lambda x: clean_from_urls(x))
df_cstm_data_filtered["comment_text"] = df_cstm_data_filtered["comment_text"].apply(lambda x: clean_from_urls(x))

## Get rid of nans and duplicates

In [18]:
df_us_gb_data_filtered_final = df_us_gb_data_filtered.dropna().drop_duplicates(keep="last")
df_cstm_data_filtered_final = df_cstm_data_filtered.dropna().drop_duplicates(keep="last")

## Divide each dataset for dataset with emojis and without them

In [19]:
def filter_for_emodji(row):
    if emoji.emoji_count(row["comment_text"]) + emoji.emoji_count(row["channel_title"]) + emoji.emoji_count(row["title"]) >= 1:
        return True
    else:
        return False

# filter for emodjis
cstm_has_emodji = df_cstm_data_filtered_final.apply(lambda row: filter_for_emodji(row), axis=1)
df_cstm_data_filtered_final_with_emodji = df_cstm_data_filtered_final[cstm_has_emodji]

us_gb_has_emodji = df_us_gb_data_filtered_final.apply(lambda row: filter_for_emodji(row), axis=1)
df_us_gb_data_filtered_final_with_emodji = df_us_gb_data_filtered_final[us_gb_has_emodji]

df_cstm_data_filtered_final_without_emodji = df_cstm_data_filtered_final[~cstm_has_emodji]
df_us_gb_data_filtered_final_without_emodji = df_us_gb_data_filtered_final[~us_gb_has_emodji]

## Left only needed columns and save to csv

In [20]:
# Data with emoji
df_cstm_data_filtered_final_with_emodji = df_cstm_data_filtered_final_with_emodji[["title", "channel_title", "comment_text"]]
df_us_gb_data_filtered_final_with_emodji = df_us_gb_data_filtered_final_with_emodji[["title", "channel_title", "comment_text"]]

# Data without emoji
df_cstm_data_filtered_final_without_emodji = df_cstm_data_filtered_final_without_emodji[["title", "channel_title", "comment_text"]]
df_us_gb_data_filtered_final_without_emodji = df_us_gb_data_filtered_final_without_emodji[["title", "channel_title", "comment_text"]]

# df_us_gb_data_filtered_final_with_emodji.to_csv(f"{DATA_PATH}/final_grouped_dataset_us_gb_with_emoji.csv")
# df_cstm_data_filtered_final_with_emodji.to_csv(f"{DATA_PATH}/final_grouped_dataset_cstm_with_emoji.csv")

# df_us_gb_data_filtered_final_without_emodji.to_csv(f"{DATA_PATH}/final_grouped_dataset_us_gb_without_emoji.csv")
# df_cstm_data_filtered_final_without_emodji.to_csv(f"{DATA_PATH}/final_grouped_dataset_cstm_without_emoji.csv")

In [21]:
df_us_gb_data_filtered_final_with_emodji.shape

(70375, 3)

In [22]:
df_cstm_data_filtered_final_with_emodji.shape

(227218, 3)

In [23]:
df_us_gb_data_filtered_final_without_emodji.shape

(448574, 3)

In [24]:
df_cstm_data_filtered_final_without_emodji.shape

(604361, 3)

In [25]:
df_us_gb_data_filtered_final_with_emodji

,title,channel_title,comment_text
0,1 YEAR OF VLOGGING -- HOW LOGAN PAUL CHANGED Y...,Logan Paul Vlogs,Logan Paul it's yo big day ‼️‼️‼️
6,1 YEAR OF VLOGGING -- HOW LOGAN PAUL CHANGED Y...,Logan Paul Vlogs,The end though 😭👍🏻❤️
24,1 YEAR OF VLOGGING -- HOW LOGAN PAUL CHANGED Y...,Logan Paul Vlogs,I love Logan and Jake so much and thay are so ...
30,1 YEAR OF VLOGGING -- HOW LOGAN PAUL CHANGED Y...,Logan Paul Vlogs,👈 watch by clicking here you can see people's ...
35,1 YEAR OF VLOGGING -- HOW LOGAN PAUL CHANGED Y...,Logan Paul Vlogs,"Evan is being a douch Logans getting pissed, L..."
...,...,...,...
628716,ZHU - Waters of Monaco (Adidas China Pure Edit),ZHUVEVO,Let's face it! Your the best zhu! 😘
628739,Destiny 2: Wardcliff Coil Glitch! How To Get U...,Kota,The Road is long and hard but we will make it ...
628741,Destiny 2: Wardcliff Coil Glitch! How To Get U...,Kota,Doesn't know if this is going to be useful for...
628806,World's First Prestige Leviathan Raid Completi...,Gladd,Get ready for all the people complaining about...
